<a href="https://colab.research.google.com/github/YANHONGLU/Financial-Crime-Data-Analytics-Projects/blob/main/Financial_Crime_Data_Analytics_Projects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
from sklearn.metrics import roc_auc_score

#Part 1. FinCrime Data Quality Checks

Business Scenario

Before running financial crime monitoring, clean the transaction data by removing duplicates, missing key fields and invalid transaction amounts.

In [16]:
#table "customer_id", "date", "amount", "partner"
def clean_transaction_data(transactions):

    # Remove duplicate rows
    clean_data = transactions.drop_duplicates()

    # Remove rows with missing key fields
    clean_data = clean_data.dropna(subset=["customer_id", "date", "amount", "partner"])

    # Keep only valid positive amounts
    clean_data = clean_data[clean_data["amount"] > 0 ]

    # Convert date to datetime format
    clean_data["date"] = pd.to_datetime( clean_data["date"] )

    return clean_data

#Part 2. Transaction Monitoring & Threshold Tuning

Business Scenario

Find suspicious customers using fixed rules and recent transaction behaviour.

In [17]:
#table:"customer_id", "date"
def monitor_transactions(transactions):
    # Group transactions by customer and date ⭐
    daily=transactions.groupby(["customer_id","date"],as_index=False).agg(daily_amount=("amount","sum"),high_risk_count=("high_risk","sum")) #Grouping field or regular column
    # Sort by customer and date
    daily = daily.sort_values(["customer_id", "date"])

    # Calculate the previous 7-day average amount
    daily["rolling_avg_7d"] =daily.groupby("customer_id")["daily_amount"].transform(lambda x: x.shift(1).rolling(7,min_periods=1).mean() ) #min_periods  must be at least a few data points before calculation begins.
    # Flag large amounts or many high-risk transactions⭐
    daily["rule_alert"] = ((daily["daily_amount"] > 10000) |(daily["high_risk_count"] >= 3) )

    # Flag amounts over 3 times the recent average
    daily["threshold_alert"] = daily["daily_amount"] > daily["rolling_avg_7d"] * 3

    # Keep transactions flagged by either rule
    suspicious = daily[ daily["rule_alert"] |daily["threshold_alert"]  ]

    return suspicious

#Part 3. Partner Risk Monitoring

Business Scenario

Compare transaction activity across different partners and identify partners with unusually high levels of high-risk transactions.

In [18]:
# table "partner"  "amount" "high_risk"
def monitor_partner_risk(transactions):

    # Group transactions by partner
    partner_summary=transactions.groupby("partner").agg(total_transactions=("amount", "count"),total_amount=("amount", "sum"),high_risk_transactions=("high_risk", "sum"))

    # Calculate the high-risk transaction rate
    partner_summary["high_risk_rate"]=partner_summary["high_risk_transactions"]/partner_summary["total_transactions"]

    # Keep partners with risk rate above 5%
    high_risk_partners = partner_summary[partner_summary["high_risk_rate"] > 0.05]

    return high_risk_partners

#Part 4. FinCrime KRI Monitoring

Business Scenario

Monitor the monthly high-risk transaction rate and trigger an alert when the risk indicator increases by more than 20% compared with the previous month.

In [19]:
def monitor_fincrime_kri(transactions):

    # Calculate monthly transaction numbers  resample("ME", on="date")Grouped by the end of each month
    monthly=transactions.resample("ME",on="date").agg(total_transactions=("amount", "count"), high_risk_transactions=("high_risk", "sum")  )

    # Calculate the monthly high-risk rate
    monthly["high_risk_rate"] = (monthly["high_risk_transactions"]/monthly["total_transactions"])

    # Calculate the change from last month
    monthly["monthly_change"] = monthly["high_risk_rate"].pct_change()

    # Alert if the rate increases by more than 20%
    monthly["alert"] = monthly["monthly_change"] > 0.20

    return monthly

# Part 5. Model Performance Monitoring

Business Scenario

Last month's model AUC was 0.80. Calculate the current AUC and trigger an alert if model performance decreases by more than 0.05.

In [20]:
def check_model(y_true, probabilities, previous_auc):

    # Calculate the current ROC AUC  roc_auc_score = To see how accurate the model's guesses are, and whether it can distinguish between "yes" and "no"
    current_auc = roc_auc_score(y_true,probabilities ) # y_true real result ;probabilities model's guessed answer/probability

    # Calculate the performance drop
    performance_drop = previous_auc - current_auc

    # Alert if AUC drops by more than 0.05
    alert = performance_drop > 0.05

    return current_auc, alert